# Appendix benchmark: GFR-RNN vs SNN baselines

This notebook reproduces the sequential MNIST comparison between the GFR-RNN, SNN-LIF, and SNN-Synaptic models used for the appendix.

Running the benchmark will:
- train or reload the three models with matched data and optimization settings;
- render a summary table and comparison plots inline;
- save checkpoints, a JSON summary, a CSV table, a LaTeX table, and a PNG overview figure under `paper_submission/appendix_outputs/<run_name>/`.

For a quick smoke test, lower `epochs` in the configuration cell before running the notebook.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

try:
    from compare_models import run_comparison
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"{exc}. Install the repo requirements first with `pip install -r requirements.txt`."
    ) from exc

pd.options.display.float_format = "{:.4f}".format
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
RUN_CONFIG = {
    # Use snn_hidden_dim=67 to roughly match the GFR-RNN parameter count at hidden_dim=64.
    "hidden_dim": 64,
    "snn_hidden_dim": 67,
    "epochs": 300,
    "lr": 1e-3,
    "batch_size": 128,
    "variant": "l",
    "seed": 42,
    "beta": 0.95,
    "alpha": 0.9,
}

LOAD_FROM_DISK_IF_AVAILABLE = True
RUN_NAME = (
    f"appendix_gfr_vs_snn_{RUN_CONFIG['variant']}"
    f"_gfr{RUN_CONFIG['hidden_dim']}"
    f"_snn{RUN_CONFIG['snn_hidden_dim']}"
    f"_ep{RUN_CONFIG['epochs']}"
    f"_seed{RUN_CONFIG['seed']}"
)
OUTPUT_DIR = Path("paper_submission") / "appendix_outputs" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}.json"

print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"Artifacts will be saved to: {OUTPUT_DIR}")
RUN_CONFIG

In [ ]:
if LOAD_FROM_DISK_IF_AVAILABLE and SUMMARY_PATH.exists():
    with SUMMARY_PATH.open() as handle:
        payload = json.load(handle)
    print(f"Loaded existing results from {SUMMARY_PATH}")
else:
    payload = run_comparison(save_dir=OUTPUT_DIR, run_name=RUN_NAME, **RUN_CONFIG)
    print(f"Completed a new run and saved results to {SUMMARY_PATH}")

In [ ]:
model_order = [name for name in ["GFR-RNN", "SNN-LIF", "SNN-Synaptic"] if name in payload["results"]]
summary_df = (
    pd.DataFrame.from_dict(payload["results"], orient="index")
    .reindex(model_order)
    .loc[:, ["trainable_params", "train_accuracy", "test_accuracy"]]
)
summary_df.index.name = "Model"
summary_df.columns = ["Trainable Parameters", "Train Accuracy", "Test Accuracy"]

paper_table = summary_df.copy()
paper_table["Train Accuracy"] = 100.0 * paper_table["Train Accuracy"]
paper_table["Test Accuracy"] = 100.0 * paper_table["Test Accuracy"]

csv_path = OUTPUT_DIR / "comparison_summary.csv"
latex_path = OUTPUT_DIR / "comparison_summary.tex"
paper_table.round(2).to_csv(csv_path)
with latex_path.open("w") as handle:
    handle.write(
        paper_table.round(2).to_latex(
            caption="Sequential MNIST comparison between GFR-RNN and SNN baselines.",
            label="tab:gfr_snn_comparison",
            float_format="%.2f",
        )
    )

print(f"Saved CSV summary to {csv_path}")
print(f"Saved LaTeX table to {latex_path}")
paper_table.round(2)

In [ ]:
loss_df = pd.DataFrame({name: payload["results"][name]["losses"] for name in model_order})
loss_df.index = loss_df.index + 1

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

for model_name in model_order:
    axes[0].plot(loss_df.index, loss_df[model_name], linewidth=2, label=model_name)
axes[0].set_title("Training loss across epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Epoch loss")
axes[0].set_yscale("log")
axes[0].legend(frameon=False)

x = np.arange(len(model_order))
bar_width = 0.35
axes[1].bar(x - bar_width / 2, paper_table.loc[model_order, "Train Accuracy"], width=bar_width, label="Train")
axes[1].bar(x + bar_width / 2, paper_table.loc[model_order, "Test Accuracy"], width=bar_width, label="Test")
axes[1].set_title("Accuracy comparison")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_order, rotation=15)
axes[1].set_ylim(0, 100)
axes[1].legend(frameon=False)

overview_path = OUTPUT_DIR / "comparison_overview.png"
fig.savefig(overview_path, dpi=300, bbox_inches="tight")
print(f"Saved overview figure to {overview_path}")
fig

In [ ]:
artifact_paths = {
    "summary_json": payload["artifacts"]["summary_path"],
    "summary_csv": str(csv_path),
    "summary_latex": str(latex_path),
    "overview_figure": str(overview_path),
}
artifact_paths.update(
    {f"checkpoint_{name.lower().replace('-', '_')}": path for name, path in payload["artifacts"]["checkpoint_paths"].items()}
)
pd.Series(artifact_paths, name="path").to_frame()